<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/websocket_chat_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

未検証・・

# WebSocket チャットデモ (Google Colab)

`websockets` ライブラリで **チャットサーバ** を立て、**ngrok** で公開して、
PC やスマホの複数ブラウザから同時にチャットできるデモです。

**構成**

```
[ブラウザ A] --wss--\
[ブラウザ B] --wss----> ngrok --> Colab上の websockets サーバ (1ポート)
[ブラウザ C] --wss--/                 ├ GET /      -> チャット画面(HTML)を返す
                                      └ /ws        -> WebSocket でメッセージ中継
```

- サーバはメッセージを受け取ると **接続中の全員にブロードキャスト**します。
- 同じ 1 ポートで HTML 配信と WebSocket を兼用するので、**ngrok トンネルは 1 本**で済みます（無料プラン向き）。
- 上から順にセルを実行してください。

In [ ]:
!pip install -q websockets==12.0 pyngrok

## 1. チャット画面(HTML/JS クライアント) を定義
バニラ JS・ダークテーマ・日本語UI。`window.__WS_URL__` があればそこへ、無ければ同一ホストの `/ws` へ自動接続します。

In [ ]:
HTML_PAGE = r'''<!doctype html>
<html lang="ja">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>WebSocket Chat Demo</title>
<style>
  :root{
    --bg:#0d1117; --panel:#161b22; --panel2:#1c2330;
    --border:#30363d; --text:#e6edf3; --muted:#8b949e;
    --accent:#39d3bb; --accent2:#58a6ff; --me:#1f6feb; --sys:#6e7681;
  }
  *{box-sizing:border-box}
  html,body{margin:0;height:100%}
  body{
    background:var(--bg); color:var(--text);
    font-family:"Hiragino Kaku Gothic ProN","Yu Gothic",Meiryo,system-ui,sans-serif;
    display:flex; flex-direction:column; height:100vh;
  }
  header{
    padding:10px 16px; border-bottom:1px solid var(--border);
    background:var(--panel); display:flex; align-items:center; gap:12px;
  }
  header h1{font-size:15px; margin:0; letter-spacing:.5px;
    font-family:"SFMono-Regular",Consolas,Menlo,monospace; color:var(--accent);}
  .status{font-size:12px; color:var(--muted); display:flex; align-items:center; gap:6px; margin-left:auto;}
  .dot{width:9px;height:9px;border-radius:50%;background:var(--sys);transition:background .3s}
  .dot.on{background:var(--accent)} .dot.off{background:#f85149}
  .count{font-size:12px;color:var(--accent2);font-family:monospace}

  #log{flex:1; overflow-y:auto; padding:14px 16px; display:flex; flex-direction:column; gap:8px;}
  .msg{max-width:78%; padding:8px 12px; border-radius:12px; line-height:1.5; font-size:14px;
       background:var(--panel2); border:1px solid var(--border); align-self:flex-start; word-break:break-word;}
  .msg.me{align-self:flex-end; background:var(--me); border-color:var(--me);}
  .msg .name{font-size:11px; color:var(--accent); margin-bottom:2px; font-weight:600;}
  .msg.me .name{color:#cfe1ff;}
  .msg .time{font-size:10px; color:var(--muted); margin-left:8px;}
  .msg.me .time{color:#cfe1ff;}
  .system{align-self:center; font-size:12px; color:var(--sys); background:transparent; border:none; padding:2px;}

  footer{padding:10px 12px; border-top:1px solid var(--border); background:var(--panel);
         display:flex; gap:8px; align-items:flex-end;}
  .namebox{display:flex; flex-direction:column; gap:3px;}
  .namebox label{font-size:10px;color:var(--muted)}
  input,button{font-family:inherit; font-size:14px;}
  #name{width:120px; padding:8px; border-radius:8px; border:1px solid var(--border);
        background:var(--bg); color:var(--text);}
  #text{flex:1; padding:10px 12px; border-radius:8px; border:1px solid var(--border);
        background:var(--bg); color:var(--text);}
  #text:focus,#name:focus{outline:none;border-color:var(--accent2)}
  #send{padding:10px 18px; border-radius:8px; border:none; cursor:pointer;
        background:var(--accent); color:#06231f; font-weight:700;}
  #send:disabled{opacity:.4; cursor:not-allowed}
</style>
</head>
<body>
  <header>
    <h1>&#128172; WebSocket Chat</h1>
    <div class="status">
      <span class="dot" id="dot"></span><span id="state">接続中...</span>
      <span class="count" id="count"></span>
    </div>
  </header>
  <div id="log"></div>
  <footer>
    <div class="namebox">
      <label for="name">名前</label>
      <input id="name" placeholder="名無し" maxlength="20">
    </div>
    <input id="text" placeholder="メッセージを入力 (Enter で送信)" autocomplete="off">
    <button id="send" disabled>送信</button>
  </footer>

<script>
  // 接続先: インライン表示時は window.__WS_URL__ を注入。
  // ngrok の URL を直接ブラウザで開いた場合は同一ホストへ自動接続。
  const WS_URL = window.__WS_URL__ ||
    ((location.protocol === "https:" ? "wss://" : "ws://") + location.host + "/ws");

  const log = document.getElementById("log");
  const dot = document.getElementById("dot");
  const stateEl = document.getElementById("state");
  const countEl = document.getElementById("count");
  const nameEl = document.getElementById("name");
  const textEl = document.getElementById("text");
  const sendBtn = document.getElementById("send");

  let ws, myName = "";

  function esc(s){return s.replace(/[&<>"']/g, c =>
    ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));}

  function addMsg(name, text, time, mine){
    const d = document.createElement("div");
    d.className = "msg" + (mine ? " me" : "");
    d.innerHTML = '<div class="name">'+esc(name)+'<span class="time">'+esc(time)+'</span></div>'+esc(text);
    log.appendChild(d); log.scrollTop = log.scrollHeight;
  }
  function addSys(text){
    const d = document.createElement("div");
    d.className = "system"; d.textContent = text;
    log.appendChild(d); log.scrollTop = log.scrollHeight;
  }

  function connect(){
    ws = new WebSocket(WS_URL);
    ws.onopen = () => {
      dot.className = "dot on"; stateEl.textContent = "接続済み";
      sendBtn.disabled = false;
      myName = (nameEl.value || "名無し").trim();
      ws.send(JSON.stringify({type:"join", name: myName}));
    };
    ws.onmessage = (e) => {
      const m = JSON.parse(e.data);
      if(m.type === "message") addMsg(m.name, m.text, m.time, m.name === myName);
      else if(m.type === "system") addSys(m.text + "  (" + m.time + ")");
      else if(m.type === "count") countEl.textContent = "オンライン: " + m.count;
    };
    ws.onclose = () => {
      dot.className = "dot off"; stateEl.textContent = "切断 — 3秒後に再接続";
      sendBtn.disabled = true; setTimeout(connect, 3000);
    };
    ws.onerror = () => ws.close();
  }

  function send(){
    const t = textEl.value.trim();
    if(!t || !ws || ws.readyState !== 1) return;
    const n = (nameEl.value || "名無し").trim();
    if(n !== myName){ myName = n; }
    ws.send(JSON.stringify({type:"chat", name: myName, text: t}));
    textEl.value = ""; textEl.focus();
  }

  sendBtn.onclick = send;
  textEl.addEventListener("keydown", e => { if(e.key === "Enter") send(); });
  connect();
</script>
</body>
</html>
'''
print('HTML クライアントを読み込みました (%d 文字)' % len(HTML_PAGE))


## 2. WebSocket サーバを起動
バックグラウンドスレッドで動かすのでセルはすぐ返ります。接続/切断/発言はこのセルの下にログ表示されます。

In [ ]:
import asyncio, json, threading, http
from datetime import datetime
import websockets

PORT = 8765
USERS = {}        # websocket -> 表示名
_started = False   # 二重起動ガード

def now():
    return datetime.now().strftime("%H:%M:%S")

def jdump(obj):
    return json.dumps(obj, ensure_ascii=False)

def push_count():
    websockets.broadcast(list(USERS.keys()), jdump({"type": "count", "count": len(USERS)}))

# --- WebSocket ハンドラ: 1接続 = 1クライアント ---
async def handler(ws):
    USERS[ws] = None
    print(f"[{now()}] 接続  (現在 {len(USERS)} 人)")
    try:
        async for raw in ws:
            try:
                data = json.loads(raw)
            except json.JSONDecodeError:
                continue
            t = data.get("type")
            if t == "join":
                USERS[ws] = (data.get("name") or "名無し").strip() or "名無し"
                websockets.broadcast(list(USERS.keys()),
                    jdump({"type": "system", "text": f"{USERS[ws]} さんが参加しました", "time": now()}))
                push_count()
            elif t == "chat":
                name = USERS.get(ws) or (data.get("name") or "名無し")
                msg = {"type": "message", "name": name,
                       "text": str(data.get("text", "")), "time": now()}
                websockets.broadcast(list(USERS.keys()), jdump(msg))
                print(f"[{now()}] {name}: {msg['text']}")
    finally:
        name = USERS.pop(ws, None)
        print(f"[{now()}] 切断  (現在 {len(USERS)} 人)")
        if name:
            websockets.broadcast(list(USERS.keys()),
                jdump({"type": "system", "text": f"{name} さんが退室しました", "time": now()}))
        push_count()

# --- 同じポートで HTML(GET /) と WebSocket(/ws) を兼用 ---
# websockets==12.0 の旧 process_request シグネチャ: (path, request_headers)
async def process_request(path, request_headers):
    if path == "/ws":
        return None                      # None を返すと WebSocket ハンドシェイクに進む
    body = HTML_PAGE.encode("utf-8")     # それ以外は HTML を返す（普通のブラウザGET）
    headers = [("Content-Type", "text/html; charset=utf-8"),
               ("Content-Length", str(len(body)))]
    return http.HTTPStatus.OK, headers, body

async def _main():
    async with websockets.serve(handler, "0.0.0.0", PORT, process_request=process_request):
        print(f"WebSocket サーバ起動: ws://0.0.0.0:{PORT}/ws")
        await asyncio.Future()           # 永久に走らせる

def _run():
    asyncio.run(_main())

if not _started:
    threading.Thread(target=_run, daemon=True).start()
    _started = True
    print("サーバをバックグラウンドスレッドで起動しました。")
else:
    print("既に起動済みです。コードを変えた場合はランタイムを再起動してください。")


## 3. ngrok で公開
初回は authtoken の設定が必要です。出力された **公開URL** を学生に配ってください。

In [ ]:
# ngrok の authtoken を設定（https://dashboard.ngrok.com/get-started/your-authtoken で取得）
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "ここに自分の authtoken を貼る"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 既存トンネルを掃除してから 1本だけ張る（無料プランは同時1本）
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(PORT, "http")
public_https = tunnel.public_url                       # 例: https://xxxx.ngrok-free.app
public_wss   = public_https.replace("https://", "wss://")
print("公開URL (学生はこれをブラウザで開く):", public_https)
print("WebSocket URL :", public_wss + "/ws")


## 4. 自分用にノートブック内で表示
まずここで動作確認。学生は手順3の公開URLを各自のブラウザ/スマホで開きます。

In [ ]:
from IPython.display import HTML, display

# インライン表示用に WS の接続先を明示注入（Colab出力はiframe隔離のため location.host が使えない）
inline = ('<script>window.__WS_URL__="' + public_wss + '/ws";</script>') + HTML_PAGE
display(HTML(inline))


## 使い方・補足

- **学生の参加**: 手順3で出た `https://....ngrok-free.app` を各自のブラウザで開くだけ。名前を入れて発言すると全員に届きます。
- **ngrok 無料プランの警告ページ**: 初回アクセス時に「Visit Site」の確認画面が出ます。一度クリックすれば通過します。
- **複数人テスト**: 自分のスマホとPCで同じURLを開けば 1人でも複数接続を確認できます。
- **作り直すとき**: サーバのコードを変更したら、メニューの「ランタイム → ランタイムを再起動」してから上から実行し直してください（ポート二重起動を避けるため）。
- **プロトコル観察**: ブラウザの開発者ツール → Network → WS を開くと、`join` / `chat` の JSON フレームが流れる様子を確認できます。授業で見せると分かりやすいです。

### メッセージ形式 (JSON)
| 向き | type | 内容 |
|---|---|---|
| client → server | `join` | `{type, name}` 入室 |
| client → server | `chat` | `{type, name, text}` 発言 |
| server → clients | `message` | `{type, name, text, time}` 発言の中継 |
| server → clients | `system` | `{type, text, time}` 入退室通知 |
| server → clients | `count` | `{type, count}` 接続人数 |